# EDA: Detekcja phishingu

Eksploracyjna analiza danych zebranych ze zbiorów:
- SpamAssassin Public Corpus
- Kampanie Gophish/MailHog (dane własne)
- Enron Email Dataset (opcjonalnie)

**Cel:** zrozumienie rozkładu cech przed treningiem klasyfikatora.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.notebook import tqdm

from agent.ingestion.dataset_loader import load_all
from agent.ingestion.email_parser import ParsedEmail
from agent.features.extractor import extract, NUMERICAL_FEATURE_COLS

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('Setup OK')

## 1. Ładowanie danych

In [ ]:
df_raw = load_all(max_per_class=2000)
print(f'Łącznie: {len(df_raw)} wiadomości')
df_raw['label_name'] = df_raw['label'].map({0: 'legit', 1: 'phishing/spam'})
df_raw.head(3)

## 2. Rozkład klas i źródeł

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rozkład klas
class_counts = df_raw['label_name'].value_counts()
axes[0].bar(class_counts.index, class_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Rozkład klas')
axes[0].set_ylabel('Liczba wiadomości')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Rozkład źródeł
source_counts = df_raw['source'].str.split('/').str[0].value_counts()
axes[1].barh(source_counts.index, source_counts.values, color='mediumseagreen')
axes[1].set_title('Źródła danych')
axes[1].set_xlabel('Liczba wiadomości')

plt.tight_layout()
plt.show()
print(f'\nImbalance ratio: {class_counts.max()/class_counts.min():.2f}x')

## 3. Ekstrakcja cech

In [ ]:
print('Parsowanie i ekstrakcja cech...')
rows = []
for _, row in tqdm(df_raw.iterrows(), total=len(df_raw)):
    try:
        parsed = ParsedEmail.from_string(row['raw_eml'])
        feats = extract(parsed)
        feats['label'] = row['label']
        feats['label_name'] = row['label_name']
        rows.append(feats)
    except Exception:
        pass

df = pd.DataFrame(rows)
num_cols = [c for c in NUMERICAL_FEATURE_COLS if c in df.columns]
df[num_cols] = df[num_cols].fillna(0)
print(f'DataFrame cech: {df.shape[0]} wierszy × {df.shape[1]} kolumn')

## 4. Rozkłady cech numerycznych (legit vs phishing)

In [ ]:
# Top cechy wg różnicy średnich (znormalizowanej)
legit   = df[df['label'] == 0][num_cols]
phish   = df[df['label'] == 1][num_cols]
diff    = (phish.mean() - legit.mean()).abs().sort_values(ascending=False)
top_feats = diff.head(12).index.tolist()

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(top_feats):
    ax = axes[i]
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        vals = df[df['label'] == label][col].clip(upper=df[col].quantile(0.99))
        ax.hist(vals, bins=30, alpha=0.6, color=color,
                label='legit' if label == 0 else 'phishing')
    short = col.replace('feat_', '').replace('nlp_', 'nlp/')
    ax.set_title(short, fontsize=9)
    ax.legend(fontsize=7)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}'))

plt.suptitle('Top 12 cech numerycznych (różnica legit vs phishing)', y=1.01)
plt.tight_layout()
plt.show()

## 5. Macierz korelacji cech

In [ ]:
# Korelacja z etykietą
corr_with_label = df[num_cols + ['label']].corr()['label'].drop('label').sort_values()

fig, ax = plt.subplots(figsize=(6, 10))
colors = ['tomato' if v > 0 else 'steelblue' for v in corr_with_label.values]
ax.barh(corr_with_label.index, corr_with_label.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Korelacja cech z etykietą (1=phishing)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 6. TF-IDF: najważniejsze słowa per klasa

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from agent.features.text_preprocessor import NLTKTextPreprocessor

prep = NLTKTextPreprocessor()
texts_legit = df[df['label'] == 0]['text'].fillna('').tolist()
texts_phish = df[df['label'] == 1]['text'].fillna('').tolist()

def top_tfidf_words(texts, n=20):
    processed = prep.fit_transform(texts)
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    X = vec.fit_transform(processed)
    scores = X.mean(axis=0).A1
    idx = scores.argsort()[::-1][:n]
    return [(vec.get_feature_names_out()[i], scores[i]) for i in idx]

print('Obliczanie TF-IDF...')
top_legit = top_tfidf_words(texts_legit[:1000])
top_phish = top_tfidf_words(texts_phish[:1000])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title, color in [
    (axes[0], top_legit, 'Top 20 – Legit', 'steelblue'),
    (axes[1], top_phish, 'Top 20 – Phishing/Spam', 'tomato'),
]:
    words, scores = zip(*data)
    ax.barh(list(reversed(words)), list(reversed(scores)), color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Średni TF-IDF')

plt.tight_layout()
plt.show()

## 7. Analiza URL-i

In [ ]:
url_cols = [c for c in num_cols if 'url' in c]

url_summary = df.groupby('label_name')[url_cols].mean().T
url_summary.columns.name = None

ax = url_summary.plot(kind='bar', figsize=(12, 5), color=['steelblue', 'tomato'])
ax.set_title('Średnie wartości cech URL (legit vs phishing)')
ax.set_ylabel('Wartość')
ax.set_xticklabels([c.replace('feat_', '') for c in url_cols], rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 8. Cechy NLP (spaCy)

In [ ]:
nlp_cols = [c for c in num_cols if 'nlp' in c]

if nlp_cols and df[nlp_cols].sum().sum() > 0:
    nlp_summary = df.groupby('label_name')[nlp_cols].mean().T
    nlp_summary.columns.name = None

    ax = nlp_summary.plot(kind='bar', figsize=(14, 5), color=['steelblue', 'tomato'])
    ax.set_title('Średnie wartości cech NLP (legit vs phishing)')
    ax.set_ylabel('Wartość')
    ax.set_xticklabels(
        [c.replace('feat_nlp_', '') for c in nlp_cols],
        rotation=35, ha='right'
    )
    plt.tight_layout()
    plt.show()
else:
    print('Cechy NLP = 0. Upewnij się że spaCy jest zainstalowane:')
    print('  python -m spacy download en_core_web_sm')

## 9. Podsumowanie statystyczne

In [ ]:
summary = df.groupby('label_name')[num_cols].mean().T
summary['diff_abs'] = (summary.get('phishing/spam', 0) - summary.get('legit', 0)).abs()
summary = summary.sort_values('diff_abs', ascending=False)

pd.set_option('display.float_format', '{:.4f}'.format)
display(summary.head(20))